In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Hiển thị đầy đủ cột
pd.set_option("display.max_columns", None)

# Đường dẫn file gốc
RAW_PATH = "superstore.csv"
df_raw = pd.read_csv(RAW_PATH, encoding="utf-8")

print("Số dòng, số cột:", df_raw.shape)
print("\nDanh sách cột:")
print(df_raw.columns.tolist())
df_raw.head()

In [ ]:
df_raw.info()

In [ ]:
column_summary = pd.DataFrame({
    "Column": df_raw.columns,
    "Initial dtype": df_raw.dtypes.astype(str).values,
    "Non-null count": df_raw.notnull().sum().values,
    "Missing count": df_raw.isnull().sum().values,
    "Unique values": df_raw.nunique().values
})

column_summary

In [ ]:
missing_by_column = df_raw.isnull().sum()
total_missing = missing_by_column.sum()
columns_with_missing = (missing_by_column > 0).sum()

print("Số dòng dữ liệu:", df_raw.shape[0])
print("Số cột dữ liệu:", df_raw.shape[1])
print("Số cột có giá trị thiếu:", columns_with_missing)
print("Tổng số ô bị thiếu dữ liệu:", total_missing)

print("\nChi tiết số giá trị thiếu theo từng cột:")
print(missing_by_column)

In [ ]:
duplicate_rows = df_raw.duplicated().sum()

unique_order_id = df_raw["Order.ID"].nunique()
total_rows = len(df_raw)
duplicated_order_id = df_raw["Order.ID"].duplicated().sum()

print("Số dòng dữ liệu:", total_rows)
print("Số dòng trùng hoàn toàn:", duplicate_rows)
print("Số lượng Order.ID duy nhất:", unique_order_id)
print("Số dòng có Order.ID bị lặp:", duplicated_order_id)

In [ ]:
print("Kiểu dữ liệu ban đầu:")
print(df_raw.dtypes)

In [ ]:
quality_checks = {
    "Sales <= 0": (df_raw["Sales"] <= 0).sum(),
    "Quantity <= 0": (df_raw["Quantity"] <= 0).sum(),
    "Discount < 0": (df_raw["Discount"] < 0).sum(),
    "Discount > 1": (df_raw["Discount"] > 1).sum(),
    "Shipping.Cost < 0": (df_raw["Shipping.Cost"] < 0).sum(),
    "Profit < 0": (df_raw["Profit"] < 0).sum()
}

pd.DataFrame(list(quality_checks.items()), columns=["Tiêu chí kiểm tra", "Số dòng"])

In [ ]:
df_clean = df_raw.copy()

df_clean["Order.Date"] = pd.to_datetime(df_clean["Order.Date"], errors="coerce")
df_clean["Ship.Date"] = pd.to_datetime(df_clean["Ship.Date"], errors="coerce")

numeric_cols = ["Sales", "Profit", "Discount", "Quantity", "Shipping.Cost", "Year", "weeknum"]

for col in numeric_cols:
    df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

print(df_clean[["Order.Date", "Ship.Date", "Sales", "Profit", "Discount", "Quantity", "Shipping.Cost", "Year", "weeknum"]].dtypes)

In [ ]:
sales_zero_rows = df_clean[df_clean["Sales"] <= 0]

sales_zero_rows
df_clean = df_clean[df_clean["Sales"] > 0].copy()

print("Số dòng sau khi loại Sales <= 0:", len(df_clean))

In [ ]:
# Chuẩn hóa tên cột để tránh lỗi khi xử lý bằng Spark
df_clean.columns = [
    col.replace(".", "_")
    for col in df_clean.columns
]
print(df_clean.columns.tolist())

In [ ]:
print("Số giá trị khác nhau của Row_ID:", df_clean["Row_ID"].nunique())
print("\nPhân phối giá trị của cột 记录数:")
print(df_clean["记录数"].value_counts())
print("\nCác cặp Market và Market2:")
print(df_clean[["Market", "Market2"]].drop_duplicates().sort_values(["Market", "Market2"]))

In [ ]:
drop_cols = ["Row_ID", "Market2", "记录数"]

df_clean = df_clean.drop(columns=drop_cols)

print("Số dòng, số cột sau khi drop:", df_clean.shape)
print(df_clean.columns.tolist())

In [ ]:
import csv
import random
from datetime import timedelta
import pandas as pd

OUTPUT_PATH = "G10_dataset.csv"
TARGET_ROWS = 500000
RANDOM_SEED = 10

random.seed(RANDOM_SEED)

MIN_DATE = pd.Timestamp("2011-01-01")
MAX_DATE = pd.Timestamp("2014-12-31")

df_base = df_clean.copy()

df_base["Order_Date"] = pd.to_datetime(df_base["Order_Date"], errors="coerce")
df_base["Ship_Date"] = pd.to_datetime(df_base["Ship_Date"], errors="coerce")

numeric_cols = ["Sales", "Profit", "Discount", "Quantity", "Shipping_Cost", "Year", "weeknum"]
for c in numeric_cols:
    df_base[c] = pd.to_numeric(df_base[c], errors="coerce")

df_base = df_base.dropna().copy()

columns = df_base.columns.tolist()

TRANSACTION_BEHAVIOR_RULES = {
    "recent_frequent": {
        "weight": 0.12,
        "orders_range": (10, 20),
        "gap_range": (20, 60),
        "last_order_recency_range": (0, 60),
        "sales_factor": (1.15, 1.60),
        "discount_range": (0.00, 0.22),
        "margin_shift": 0.05
    },
    "regular_value": {
        "weight": 0.58,
        "orders_range": (5, 11),
        "gap_range": (70, 160),
        "last_order_recency_range": (100, 280),
        "sales_factor": (0.85, 1.25),
        "discount_range": (0.00, 0.35),
        "margin_shift": 0.00
    },
    "low_activity": {
        "weight": 0.22,
        "orders_range": (2, 4),
        "gap_range": (180, 420),
        "last_order_recency_range": (420, 850),
        "sales_factor": (0.50, 0.95),
        "discount_range": (0.00, 0.35),
        "margin_shift": -0.04
    },
    "discount_driven": {
        "weight": 0.08,
        "orders_range": (3, 7),
        "gap_range": (90, 220),
        "last_order_recency_range": (160, 420),
        "sales_factor": (0.70, 1.15),
        "discount_range": (0.30, 0.60),
        "margin_shift": -0.10
    }
}


def clamp(value, min_value, max_value):
    return max(min_value, min(value, max_value))


def calculate_weeknum(date_value):
    jan1 = pd.Timestamp(date_value.year, 1, 1)
    if jan1.weekday() == 6:
        return int(date_value.strftime("%U"))
    return int(date_value.strftime("%U")) + 1


def format_date(date_value):
    return pd.Timestamp(date_value).strftime("%Y-%m-%d")


def sample_behavior_rule():
    names = list(TRANSACTION_BEHAVIOR_RULES.keys())
    weights = [TRANSACTION_BEHAVIOR_RULES[name]["weight"] for name in names]
    selected_name = random.choices(names, weights=weights, k=1)[0]
    return selected_name, TRANSACTION_BEHAVIOR_RULES[selected_name]


def generate_customer_id(counter):
    return f"GEN-CUST-{counter:06d}"


def generate_customer_name(counter):
    return f"Generated Customer {counter:06d}"


def generate_order_id(counter):
    return f"GEN-ORD-{counter:07d}"


def generate_order_dates(rule):
    n_orders = random.randint(*rule["orders_range"])
    gap_min, gap_max = rule["gap_range"]

    recency_min, recency_max = rule["last_order_recency_range"]
    max_possible_recency = (MAX_DATE - MIN_DATE).days

    recency_days = random.randint(
        recency_min,
        min(recency_max, max_possible_recency)
    )

    last_order_date = MAX_DATE - timedelta(days=recency_days)

    order_dates = [last_order_date]
    current_date = last_order_date

    for _ in range(n_orders - 1):
        gap_days = random.randint(gap_min, gap_max)
        current_date = current_date - timedelta(days=gap_days)

        if current_date < MIN_DATE:
            break

        order_dates.append(current_date)

    # Hạn chế trường hợp khách hàng chỉ có 1 đơn nếu rule yêu cầu từ 2 đơn trở lên
    min_orders = rule["orders_range"][0]

    retry_count = 0
    while len(order_dates) < min_orders and retry_count < 5:
        retry_count += 1

        recency_days = random.randint(
            recency_min,
            min(recency_max, max_possible_recency)
        )

        last_order_date = MAX_DATE - timedelta(days=recency_days)

        temp_dates = [last_order_date]
        current_date = last_order_date

        for _ in range(n_orders - 1):
            gap_days = random.randint(gap_min, gap_max)
            current_date = current_date - timedelta(days=gap_days)

            if current_date < MIN_DATE:
                break

            temp_dates.append(current_date)

        order_dates = temp_dates

    return sorted(order_dates)


def generate_ship_date(order_date, ship_mode):
    if ship_mode == "Same Day":
        lead_time = random.randint(0, 1)
    elif ship_mode == "First Class":
        lead_time = random.randint(1, 3)
    elif ship_mode == "Second Class":
        lead_time = random.randint(2, 5)
    else:
        lead_time = random.randint(4, 10)

    ship_date = order_date + timedelta(days=lead_time)

    if ship_date > MAX_DATE:
        ship_date = MAX_DATE

    if ship_date < order_date:
        ship_date = order_date

    return ship_date


def sample_order_line_count():
    return random.choices(
        population=[1, 2, 3, 4, 5],
        weights=[0.38, 0.30, 0.18, 0.10, 0.04],
        k=1
    )[0]


def calculate_profit_margin(base_sales, base_profit, discount, rule):
    if base_sales <= 0:
        base_margin = 0.10
    else:
        base_margin = base_profit / base_sales

    base_margin = clamp(base_margin, -0.50, 0.60)

    discount_penalty = discount * 0.45
    random_noise = random.uniform(-0.06, 0.06)

    profit_margin = (
        base_margin
        + rule["margin_shift"]
        - discount_penalty
        + random_noise
    )

    return clamp(profit_margin, -0.80, 0.70)


customer_context_cols = [
    "City", "Country", "Market", "Region", "State", "Segment"
]

product_cols = [
    "Product_ID", "Product_Name", "Category", "Sub_Category"
]

order_level_cols = [
    "Order_Priority", "Ship_Mode"
]

customer_context_pool = (
    df_base[customer_context_cols]
    .drop_duplicates()
    .to_dict("records")
)

product_pool = (
    df_base[
        product_cols
        + ["Sales", "Profit", "Quantity", "Discount", "Shipping_Cost"]
    ]
    .dropna()
    .to_dict("records")
)

order_priority_values = df_base["Order_Priority"].dropna().unique().tolist()
ship_mode_values = df_base["Ship_Mode"].dropna().unique().tolist()

current_rows = 0
generated_customers = 0
generated_orders = 0
generated_lines = 0

customer_counter = 1
order_counter = 1

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(columns)

    # 1. Ghi dữ liệu sau làm sạch trước
    for _, row in df_base.iterrows():
        output_row = []

        for col in columns:
            value = row[col]

            if col in ["Order_Date", "Ship_Date"]:
                value = format_date(value)

            output_row.append(value)

        writer.writerow(output_row)
        current_rows += 1

    # 2. Sinh thêm dữ liệu theo cấu trúc customer -> order -> order-line
    while current_rows < TARGET_ROWS:
        behavior_name, rule = sample_behavior_rule()

        customer_id = generate_customer_id(customer_counter)
        customer_name = generate_customer_name(customer_counter)
        customer_counter += 1
        generated_customers += 1

        customer_context = random.choice(customer_context_pool)
        order_dates = generate_order_dates(rule)

        for order_date in order_dates:
            if current_rows >= TARGET_ROWS:
                break

            order_id = generate_order_id(order_counter)
            order_counter += 1
            generated_orders += 1

            order_priority = random.choice(order_priority_values)
            ship_mode = random.choice(ship_mode_values)
            ship_date = generate_ship_date(order_date, ship_mode)

            line_count = sample_order_line_count()

            for _ in range(line_count):
                if current_rows >= TARGET_ROWS:
                    break

                product_template = random.choice(product_pool)

                base_sales = float(product_template["Sales"])
                base_profit = float(product_template["Profit"])
                base_quantity = int(product_template["Quantity"])
                base_shipping = float(product_template["Shipping_Cost"])

                sales_factor = random.uniform(*rule["sales_factor"])
                discount = random.uniform(*rule["discount_range"])
                discount = clamp(discount, 0, 0.85)

                quantity = base_quantity + random.choice([-1, 0, 1])
                quantity = int(clamp(quantity, 1, 14))

                base_unit_sales = base_sales / max(base_quantity, 1)
                sales = base_unit_sales * quantity * sales_factor
                sales = round(max(sales, 0.01), 2)

                profit_margin = calculate_profit_margin(
                    base_sales=base_sales,
                    base_profit=base_profit,
                    discount=discount,
                    rule=rule
                )

                profit = round(sales * profit_margin, 2)

                shipping_factor = random.uniform(0.75, 1.35)
                shipping_cost = max(base_shipping * sales_factor * shipping_factor, 0)
                shipping_cost = round(shipping_cost, 2)

                new_row = {col: None for col in columns}

                # Customer and market context
                new_row["Customer_ID"] = customer_id
                new_row["Customer_Name"] = customer_name
                for col in customer_context_cols:
                    new_row[col] = customer_context[col]

                # Order information
                new_row["Order_ID"] = order_id
                new_row["Order_Date"] = order_date
                new_row["Ship_Date"] = ship_date
                new_row["Order_Priority"] = order_priority
                new_row["Ship_Mode"] = ship_mode

                # Product information
                for col in product_cols:
                    new_row[col] = product_template[col]

                # Numeric information
                new_row["Sales"] = sales
                new_row["Profit"] = profit
                new_row["Quantity"] = quantity
                new_row["Discount"] = round(discount, 2)
                new_row["Shipping_Cost"] = shipping_cost
                new_row["Year"] = order_date.year
                new_row["weeknum"] = calculate_weeknum(order_date)

                output_row = []

                for col in columns:
                    value = new_row[col]

                    if col in ["Order_Date", "Ship_Date"]:
                        value = format_date(value)

                    output_row.append(value)

                writer.writerow(output_row)

                current_rows += 1
                generated_lines += 1

print(f"Đã tạo thành công file {OUTPUT_PATH}")
print(f"Tổng số dòng: {current_rows}")
print(f"Số khách hàng sinh thêm: {generated_customers}")
print(f"Số đơn hàng sinh thêm: {generated_orders}")
print(f"Số dòng sinh thêm: {generated_lines}")

In [ ]:
df_final = pd.read_csv("G10_dataset.csv", encoding="utf-8")

print("Số dòng, số cột:", df_final.shape)
print("Số Customer_ID:", df_final["Customer_ID"].nunique())
print("Số Order_ID:", df_final["Order_ID"].nunique())
print("Số Product_ID:", df_final["Product_ID"].nunique())

avg_lines_per_order = len(df_final) / df_final["Order_ID"].nunique()
avg_orders_per_customer = (
    df_final.groupby("Customer_ID")["Order_ID"].nunique().mean()
)

print("Số dòng trung bình / đơn hàng:", round(avg_lines_per_order, 2))
print("Số đơn hàng trung bình / khách hàng:", round(avg_orders_per_customer, 2))